# 22104822 Delabie Walther
# 22100961 Paysant Clementine
# 22002484 Fauquette Maël

In [ ]:

import torch
import torch.nn.functional as F
import datasets
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from diffusers import UNet2DModel , DDPMScheduler
from diffusers.optimization import get_cosine_schedule_with_warmup
from datasets import load_dataset


# Projet : Outpainting d’images par Diffusion (DDPM)

## Objectif du projet

Ce projet vise à implémenter un **modèle de diffusion conditionnel** capable de réaliser de l’**outpainting**, c’est-à-dire **générer les parties manquantes d’une image** (ici les bords gauche et droit) à partir d’une image partiellement observée.

Le modèle apprend à reconstruire les zones masquées en s’appuyant sur :
- le contenu visible de l’image (condition),
- un processus de diffusion de type **DDPM (Denoising Diffusion Probabilistic Model)**,
- une architecture **U-Net**.

---

## 1. Configuration générale

Les principaux hyperparamètres du projet sont :
- Taille des images : `16 × 16`
- Taille de batch : `16`
- Nombre d’époques : `5`
- Taux d’apprentissage : `1e-4`

L’entraînement est effectué sur **GPU si disponible**, sinon sur CPU.

---

## 2. Préparation des données

### Chargement du dataset

Le dataset est chargé à partir d’un dossier d’images via `datasets.load_dataset("imagefolder")`.

### Prétraitements

Chaque image subit les transformations suivantes :
- Redimensionnement en `16 × 16`
- Conversion en tenseur PyTorch
- Normalisation dans l’intervalle `[-1, 1]`

### Création de la tâche d’outpainting

Pour simuler une image partielle :
- Le **centre de l’image est conservé**
- Les **bords gauche et droit (1/4 de la largeur chacun)** sont masqués
- Les zones masquées sont remplacées par `-1` (noir après normalisation)

Cette image partielle sert de **condition** au modèle.

---

## 3. Architecture du modèle

Le modèle repose sur un **U-Net 2D**, utilisé classiquement pour les modèles de diffusion.

### Particularité du modèle

- **Entrée : 6 canaux**
  - 3 canaux : image bruitée
  - 3 canaux : image conditionnelle (image masquée)
- **Sortie : 3 canaux**
  - prédiction du bruit ajouté à l’image cible

L’architecture inclut :
- plusieurs blocs de downsampling et upsampling,
- des blocs avec **self-attention** pour capturer des dépendances globales.

---

# 4. Modèle de diffusion (DDPM)

Les modèles de diffusion reposent sur deux processus complémentaires :  
- un **processus de diffusion directe** (ajout de bruit)  
- un **processus inverse de débruitage** (génération).

Le bruit est géré par un **DDPMScheduler**, caractérisé par :  

- un nombre fixe de pas de diffusion \(T\) (typiquement \(T = 1000\))  
- une **noise schedule** définissant la variance du bruit à chaque pas  
- un processus inverse appris par un réseau de neurones (U-Net)  

---

## Paramètres de diffusion

On définit une suite de variances telle que :

$$
\beta_t \in (0, 1)
$$

et :

$$
\alpha_t = 1 - \beta_t
$$

$$
\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s
$$

---

## Processus de diffusion directe (Forward Process)

Le processus direct ajoute progressivement du bruit gaussien à une image propre $x_0$.  
Il est possible d’échantillonner directement un état bruité $x_t$ à partir de $x_0$ :

$$
x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon, 
\quad \epsilon \sim \mathcal{N}(0, \mathbf{I})
$$

Cette formulation permet d’éviter de simuler explicitement tous les pas intermédiaires.

---

## Processus inverse de débruitage (Reverse Denoising Process)

Le processus inverse cherche à approximer la distribution conditionnelle :

$$
p_\theta(x_{t-1} \mid x_t) 
= \mathcal{N}\big(x_{t-1}; \mu_\theta(x_t, t), \sigma_t^2 \mathbf{I}\big)
$$

où la variance est généralement fixée à :

$$
\sigma_t^2 = \beta_t
$$

---

## Paramétrisation de la moyenne

Ho et al. (2020) proposent de paramétrer la moyenne du processus inverse à partir d’un réseau prédisant le bruit:

$$
\mu_\theta(x_t, t) = \frac{1}{\sqrt{1-\beta_t}} 
\left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \, \epsilon_\theta(x_t, t) \right)
$$

---

## Échantillonnage du pas inverse

À chaque étape, l’image est débruitée selon :

$$
x_{t-1} = \frac{1}{\sqrt{\alpha_t}} 
\left( x_t - \frac{1 - \alpha_t}{\sqrt{1 - \bar{\alpha}_t}} \, \epsilon_\theta(x_t, t) \right)
+ \sigma_t z, \quad z \sim \mathcal{N}(0, \mathbf{I})
$$

avec $z = 0$ lorsque $t = 0$.  

Ce processus permet d’inverser progressivement la diffusion jusqu’à obtenir un échantillon final $x_0$.

---

## Objectif d’entraînement

Le modèle est entraîné à prédire le bruit ajouté à une image bruitée :

$$
\mathcal{L}_{\text{simple}} = \mathbb{E}_{x_0 \sim q(x0), \epsilon \sim \mathcal{N}(0,1), t \sim \mathcal{U}(1,T)} \Big[ \| \epsilon - \epsilon_\theta(\sqrt{\bar{\alpha}_t}x_0 +\sqrt{1-\bar{\alpha}_t} \epsilon,t ) \|^2 \Big]
$$

Cet objectif correspond à une simplification du **majorant variationnel** de la log-vraisemblance.

---

## Résumé du fonctionnement

- Le **forward process** transforme progressivement les données en bruit pur.  
- Le **reverse process**, appris par un réseau de neurones, enlève ce bruit pas à pas.  
- Le **DDPMScheduler** gère la noise schedule et les transitions entre les états $x_t$.





L’optimisation est réalisée avec **AdamW**.




In [ ]:
# --- 1. Configuration ---
IMG_SIZE = 16  # Petite taille pour un entraînement rapide sur le "mini" dataset
BATCH_SIZE = 16
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4
device = "cuda" if torch.cuda.is_available() else "cpu"



In [ ]:
# --- 2. Préparation du Dataset (Création de la tâche d'Outpainting) ---
# On charge le dataset que tu as fourni
dataset = load_dataset("imagefolder", split="train")

In [ ]:


# Transformations : On redimensionne et on normalise entre -1 et 1
preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

def transform_data(examples):
    images = [preprocess(image.convert("RGB")) for image in examples["image"]]
    return {"pixel_values": images}

dataset.set_transform(transform_data)
train_dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Fonction pour créer le masque (simuler l'input partiel)
# On garde le centre, on cache les bords gauche et droite
def create_condition_image(images):
    # images shape: [Batch, 3, H, W]
    width = images.shape[3]
    mask_width = width // 4  # On cache 1/4 à gauche et 1/4 à droite

    condition = images.clone()
    # Masquer gauche
    condition[:, :, :, :mask_width] = -1  # -1 correspond au noir après normalisation
    # Masquer droite
    condition[:, :, :, -mask_width:] = -1
    return condition

# --- 3. Architecture du Modèle (U-Net) ---
# D'après le cours, on utilise un U-Net.
# IMPORTANT : in_channels=6 car on concatène l'image bruitée (3) + la condition (3)
print("Initialisation du modèle...")
model = UNet2DModel(
    sample_size=IMG_SIZE,
    in_channels=6,  # 3 channels (image bruitée) + 3 channels (condition)
    out_channels=3, # On prédit le bruit sur les 3 canaux de l'image cible
    layers_per_block=2,
    block_out_channels=(64, 128, 128, 256),
    down_block_types=(
        "DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"
    ),
    up_block_types=(
        "AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D"
    ),
).to(device)

# --- 4. Le Scheduler (Gestion du Bruit) ---
# Le cours mentionne DDPM (Denoising Diffusion Probabilistic Models)
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


---

## 5. Boucle d’entraînement

Pour chaque batch :
1. Récupération des images originales
2. Création de l’image conditionnelle (bords masqués)
3. Ajout de bruit à l’image originale selon un pas de diffusion aléatoire
4. Concaténation :
   - image bruitée
   - image conditionnelle
5. Prédiction du bruit par le U-Net
6. Calcul de la perte :
   - **MSE entre le bruit prédit et le bruit réel**
7. Rétropropagation et mise à jour des poids

La perte moyenne est affichée à la fin de chaque époque.

---



In [ ]:
# --- 5. Boucle d'Entraînement ---
print("Début de l'entraînement...")
for epoch in range(NUM_EPOCHS):
    # INITIALISATION DES VARIABLES ICI
    epoch_loss = 0.0
    num_batches = 0

    for step, batch in enumerate(train_dataloader):
        print(f"Processing batch {step+1}/{len(train_dataloader)}", end="\r")
        clean_images = batch["pixel_values"].to(device)

        # A. Créer la condition
        condition_images = create_condition_image(clean_images)

        # B. Ajouter du bruit
        noise = torch.randn_like(clean_images)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (clean_images.shape[0],), device=device).long()
        noisy_images = noise_scheduler.add_noise(clean_images, noise, timesteps)

        # C. Concaténation (Input + Condition)
        model_input = torch.cat([noisy_images, condition_images], dim=1)

        # D. Prédiction
        noise_pred = model(model_input, timesteps).sample

        # E. Calcul de la perte
        loss = F.mse_loss(noise_pred, noise)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # CUMUL DE LA PERTE ICI
        epoch_loss += loss.item()
        num_batches += 1
        print(f"Epoch {epoch+1}, Step {step+1}, Loss: {loss.item():.5f}")
    # Calcul de la moyenne à la fin de l'époque
    avg_loss = epoch_loss / num_batches
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Average Loss: {avg_loss:.5f}")
    # --- SAUVEGARDE DES POIDS (checkpoint) ---
    torch.save(model.state_dict(), f"unet_outpainting_epoch_{epoch+1}.pth")
    noise_scheduler.save_pretrained(f"scheduler_epoch_{epoch+1}")


print("Entraînement terminé.")
# --- SAUVEGARDE FINALE ---
torch.save(model.state_dict(), "unet_outpainting_final.pth")
noise_scheduler.save_pretrained("scheduler_final")



## 6. Phase d’inférence (génération)

Pour tester le modèle :
1. Une image est sélectionnée depuis le dataset
2. Les bords sont masqués pour créer la condition
3. La génération démarre à partir de **bruit pur**
4. Le modèle applique le **processus inverse de diffusion**
5. À chaque étape :
   - l’image générée est concaténée avec la condition
   - le modèle prédit le bruit
   - le scheduler retire progressivement le bruit

---

## 7. Visualisation des résultats

Trois images sont affichées :
1. Image originale
2. Image conditionnelle (bords masqués)
3. Image générée avec extension des bords

Les tenseurs sont dénormalisés pour un affichage correct.



In [ ]:
model.load_state_dict(torch.load("unet_outpainting_final.pth", map_location=device))
model.eval()

noise_scheduler = DDPMScheduler.from_pretrained("scheduler_final")
print("Modèle et scheduler chargés pour l'inférence.")
# --- 6. Inférence (Génération d'extension) ---
# Pour tester, on prend une image, on la masque, et on demande au modèle de remplir les bords.

# Préparer une condition de test
sample_img = next(iter(train_dataloader))["pixel_values"][0:1].to(device)
test_condition = create_condition_image(sample_img) # L'image avec bords noirs

# Partir du bruit pur
generated_image = torch.randn_like(sample_img)

# Boucle de Denoising (Reverse Process)
model.eval()
with torch.no_grad():
    for t in noise_scheduler.timesteps:
        # 1. Créer l'input : Image en cours de génération + Condition fixe
        model_input = torch.cat([generated_image, test_condition], dim=1)

        # 2. Prédire le bruit
        noise_prediction = model(model_input, t, return_dict=False)[0]

        # 3. Retirer le bruit (step DDPM)
        generated_image = noise_scheduler.step(noise_prediction, t, generated_image).prev_sample

# Affichage
def show_tensor(tensor, title):
    img = tensor.cpu().squeeze().permute(1, 2, 0)
    img = (img + 1) / 2 # Denormalize
    img = img.clamp(0, 1)
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")

plt.figure(figsize=(10, 4))
plt.subplot(1, 3, 1)
show_tensor(sample_img, "Originale")
plt.subplot(1, 3, 2)
show_tensor(test_condition, "Input (Condition)")
plt.subplot(1, 3, 3)
show_tensor(generated_image, "Générée (Extension)")
plt.show()



---

## Conclusion

Ce projet démontre :
- l’utilisation d’un **modèle de diffusion conditionnel**,
- l’adaptation d’un U-Net pour l’outpainting,
- la mise en œuvre complète du pipeline :
  - préparation des données,
  - entraînement DDPM,
  - génération par débruitage progressif.

Il constitue une base solide pour explorer des variantes plus avancées :
- masques arbitraires,
- guidance plus complexe,
- modèles plus profonds ou pré-entraînés.